In [ ]:
# CELL 1: SETUP & CONFIGURATION
# ============================================================
import sys
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import correlate
import soundfile as sf
import torch
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# === EDITABLE CONFIGURATION ===
# ============================================================

# Checkpoint filename (ensure it exists in checkpoints/ folder)
CHECKPOINT_NAME = "indah_best.pth" 

# Output folder for analysis results (automatically created if not exists)
ANALYSIS_OUTPUT_DIR = "analysis_output"

# ============================================================
# SYSTEM SETUP
# ============================================================

PROJECT_ROOT = Path(".").resolve()
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / CHECKPOINT_NAME
OUTPUT_DIR = PROJECT_ROOT / ANALYSIS_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# 1. Load CONFIG from Checkpoint
if CHECKPOINT_PATH.exists():
    print(f"Loading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    CONFIG = ckpt.get("config", {})
    print("Config loaded from checkpoint.")
else:
    print(f"Checkpoint not found: {CHECKPOINT_PATH}")
    print("Using fallback config (Stereo defaults).")
    CONFIG = {
        "sample_rate": 44100,
        "chunk_duration_sec": 8,
        "base_channels": 48,
        "checkpointing": {"downsample": [True]*4, "bottleneck": True, "upsample": [True]*4}
    }

# 2. Import Model Architecture (Auto-generated by train.py)
ARCH_PATH = PROJECT_ROOT / "checkpoints" / "model_architecture.py"
if ARCH_PATH.exists():
    sys.path.insert(0, str(PROJECT_ROOT / "checkpoints"))
    from model_architecture import IndahModel, get_checkpointing_preset
    print(f"Model architecture imported from {ARCH_PATH.name}")
else:
    raise FileNotFoundError(
        f"Model architecture not found at {ARCH_PATH}.\n"
        "Please run training first to generate 'model_architecture.py' in the checkpoints folder."
    )

# 3. Helper Functions
# ============================================================

def trim_to_match_length(y1, y2):
    """Trim both arrays to the same length (minimum of the two)"""
    min_len = min(y1.shape[-1], y2.shape[-1])
    return y1[..., :min_len], y2[..., :min_len]

def match_rms(y1, y2):
    """Match RMS of y2 to y1 (Stereo compatible)"""
    if y1.ndim == 1: y1 = np.vstack([y1, y1])
    if y2.ndim == 1: y2 = np.vstack([y2, y2])
    
    for ch in range(y1.shape[0]):
        rms1 = np.sqrt(np.mean(y1[ch]**2))
        rms2 = np.sqrt(np.mean(y2[ch]**2))
        if rms1 < 0.001 or rms2 < 0.001: continue
        if rms2 > 1e-6:
            y2[ch] = y2[ch] * (rms1 / rms2)
    return y1, y2

def align_audio_by_cross_correlation(ref_audio, target_audio_to_align, sr):
    """Align target to reference using cross-correlation (Stereo compatible)"""
    ref_mono = ref_audio.mean(axis=0)
    target_mono = target_audio_to_align.mean(axis=0)
    
    corr = correlate(ref_mono, target_mono, mode='full', method='fft')
    delay_samples = corr.argmax() - (len(target_mono) - 1)
    
    # Logging
    print(f"  Delay detected: {delay_samples} samples ({delay_samples/sr:.3f} seconds)")

    if delay_samples > 0:
        aligned_ref = ref_audio[:, delay_samples:]
        aligned_target = target_audio_to_align
        print(f"  Trimming reference: removed {delay_samples} samples from start")
    elif delay_samples < 0:
        offset = abs(delay_samples)
        aligned_target = target_audio_to_align[:, offset:]
        aligned_ref = ref_audio
        print(f"  Trimming target: removed {offset} samples from start")
    else:
        aligned_ref = ref_audio
        aligned_target = target_audio_to_align
        print("  Perfect alignment (no trimming needed)")
        
    min_len = min(aligned_ref.shape[1], aligned_target.shape[1])
    aligned_ref = aligned_ref[:, :min_len]
    aligned_target = aligned_target[:, :min_len]
    
    print(f"  Final aligned length: {min_len} samples ({min_len/sr:.2f} seconds)")
    
    return aligned_ref, aligned_target, delay_samples

print("\nSetup complete! Ready for analysis.")

In [ ]:
# CELL 2: ALIGNMENT TESTER (Updated for Stereo)
# ============================================================

# Change filename here (without .wav extension)
# Leave as None to use the first file found
TEST_FILENAME = None  # Example: "Just Enough Whiskey Late Night"

# ============================================================
# ANALYSIS FUNCTION
# ============================================================

def run_alignment_test():
    print("Starting Alignment Test...")
    
    # 1. Setup Paths based on CONFIG (from Cell 1)
    # Fallback to default names if config keys missing
    train_subdir = CONFIG.get("train_subdir", "train")
    input_name = CONFIG.get("input_dir_name", "input")
    target_name = CONFIG.get("target_dir_name", "target")
    sr = CONFIG["sample_rate"]
    
    input_dir = PROJECT_ROOT / train_subdir / input_name
    target_dir = PROJECT_ROOT / train_subdir / target_name
    
    if not input_dir.exists() or not target_dir.exists():
        raise FileNotFoundError(f"Dataset folders not found! Check: {input_dir} and {target_dir}")

    # 2. Find Paired Files
    input_files = set(f.stem for f in input_dir.glob("*.wav"))
    target_files = set(f.stem for f in target_dir.glob("*.wav"))
    paired_files = input_files & target_files
    
    if len(paired_files) == 0:
        raise FileNotFoundError("No paired .wav files found in input/target folders.")
    
    # Select File
    if TEST_FILENAME:
        if TEST_FILENAME in paired_files:
            song_name = TEST_FILENAME
            print(f"Selected file: {song_name}")
        else:
            print(f"File '{TEST_FILENAME}' not found. Using first available.")
            song_name = sorted(paired_files)[0]
    else:
        song_name = sorted(paired_files)[0]
        print(f"Using first available file: {song_name}")

    # 3. Load Audio (STEREO)
    print(f"\nLoading: {song_name}.wav")
    inp_path = input_dir / f"{song_name}.wav"
    tgt_path = target_dir / f"{song_name}.wav"
    
    inp_audio, _ = librosa.load(inp_path, sr=sr, mono=False)
    tgt_audio, _ = librosa.load(tgt_path, sr=sr, mono=False)
    
    # Ensure Stereo
    if inp_audio.ndim == 1: inp_audio = np.vstack([inp_audio, inp_audio])
    if tgt_audio.ndim == 1: tgt_audio = np.vstack([tgt_audio, tgt_audio])
    
    print(f"   Input Shape: {inp_audio.shape} | Target Shape: {tgt_audio.shape}")
    
    # Trim to match length (fix for audio files with different durations)
    inp_audio, tgt_audio = trim_to_match_length(inp_audio, tgt_audio)
    print(f"   After Trim: {inp_audio.shape} | {tgt_audio.shape}")

    # 4. Save Originals (Backup)
    sf.write(OUTPUT_DIR / f"{song_name}_original_input.wav", inp_audio.T, sr)
    sf.write(OUTPUT_DIR / f"{song_name}_original_target.wav", tgt_audio.T, sr)

    # 5. Pre-processing: RMS Match & Alignment
    print("\nProcessing...")
    inp_matched, tgt_matched = match_rms(inp_audio, tgt_audio)
    
    aligned_inp, aligned_tgt, delay = align_audio_by_cross_correlation(
        inp_matched, tgt_matched, sr
    )

    # 6. Save Aligned & Residual
    residual = aligned_tgt - aligned_inp
    
    sf.write(OUTPUT_DIR / f"{song_name}_aligned_input.wav", aligned_inp.T, sr)
    sf.write(OUTPUT_DIR / f"{song_name}_aligned_target.wav", aligned_tgt.T, sr)
    
    # 7. Analysis Metrics
    print("\nAnalysis Results:")
    
    # Correlation (Mono version for calculation)
    inp_mono = aligned_inp.mean(axis=0)
    tgt_mono = aligned_tgt.mean(axis=0)
    corr_before = np.corrcoef(inp_audio.mean(axis=0), tgt_audio.mean(axis=0))[0,1]
    corr_after = np.corrcoef(inp_mono, tgt_mono)[0,1]
    
    print(f"   Correlation (Before): {corr_before:.4f}")
    print(f"   Correlation (After):  {corr_after:.4f}")
    print(f"   Improvement:          {corr_after - corr_before:+.4f}")
    
    # Reconstruction Error
    reconstruction = aligned_inp + residual
    error = np.mean(np.abs(reconstruction - aligned_tgt))
    print(f"   Reconstruction Error: {error:.2e}")

    # 8. Plotting
    print("\nGenerating Analysis Plot...")
    plot_alignment_analysis(
        aligned_inp, aligned_tgt, residual, song_name, sr
    )
    
    print("\nAlignment Test Complete! Check 'analysis_output/' folder.")

def plot_alignment_analysis(inp, tgt, res, name, sr):
    """Generate visual comparison plots"""
    # Use only first 30 seconds for clarity
    limit = min(30 * sr, inp.shape[1])
    t = np.arange(limit) / sr
    
    fig, axes = plt.subplots(3, 2, figsize=(15, 10))
    
    # 1. Original vs Aligned (Left Channel)
    axes[0,0].plot(t, inp[0, :limit], label='Input (Aligned)', alpha=0.7)
    axes[0,0].plot(t, tgt[0, :limit], label='Target (Aligned)', alpha=0.7)
    axes[0,0].set_title(f'Aligned Waveforms (Left Ch) - {name}')
    axes[0,0].legend()
    
    # 2. Residual Waveform
    axes[0,1].plot(t, res[0, :limit], color='r')
    axes[0,1].set_title('Residual Waveform (Left Ch)')
    
    # 3. Residual Histogram
    axes[1,0].hist(res.flatten(), bins=100, color='purple', alpha=0.7)
    axes[1,0].set_title('Residual Distribution (All Channels)')
    
    # 4. RMS Residual over Time
    window = 2048
    rms_vals = []
    res_mono = res.mean(axis=0)
    for i in range(0, len(res_mono)-window, window//2):
        rms_vals.append(np.sqrt(np.mean(res_mono[i:i+window]**2)))
    
    axes[1,1].plot(np.arange(len(rms_vals))*(window/2)/sr, rms_vals, color='orange')
    axes[1,1].set_title('RMS Residual Over Time')
    
    # 5. Spectrogram (First 10s of Residual)
    plot_dur = min(10 * sr, limit)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(res[0, :plot_dur])), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=axes[2,0])
    axes[2,0].set_title('Residual Spectrogram (Left Ch)')
    
    # 6. Scatter (True vs Input)
    # Downsample for speed
    step = 100
    axes[2,1].scatter(tgt[0, :limit:step], inp[0, :limit:step], alpha=0.1, s=1)
    axes[2,1].plot(axes[2,1].get_xlim(), axes[2,1].get_xlim(), 'r--')
    axes[2,1].set_title('Target vs Input Scatter')
    
    plt.tight_layout()
    save_path = OUTPUT_DIR / f"{name}_analysis.png"
    plt.savefig(save_path, dpi=150)
    plt.show()

# Run the function
if __name__ == "__main__":
    run_alignment_test()

In [ ]:
# ============================================================
# CELL 3: RESIDUAL COMPARATOR & MODEL EVALUATION
# ============================================================


def predict_residual_stereo(model, audio_input, device, chunk_sec=8):
    """
    Runs chunk-based inference on stereo audio [2, T].
    Returns predicted residual [2, T].
    """
    model.eval()
    _, total_len = audio_input.shape
    sr = CONFIG["sample_rate"]
    chunk_len = int(chunk_sec * sr)
    chunk_len = (chunk_len // 16) * 16  # U-Net constraint

    output = np.zeros_like(audio_input)

    # Process chunks along time dimension, preserving stereo channels [2, L]
    for start in tqdm(range(0, total_len, chunk_len), desc="Processing chunks"):
        end = min(start + chunk_len, total_len)
        chunk = audio_input[:, start:end]  # Shape: [2, chunk_len]

        # Pad if last chunk is shorter
        if chunk.shape[1] < chunk_len:
            pad_len = chunk_len - chunk.shape[1]
            chunk = np.pad(chunk, ((0,0), (0, pad_len)), mode='reflect')

        # Convert to tensor: [1, 2, chunk_len] -> Matches model architecture
        inp_tensor = torch.from_numpy(chunk).float().unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(inp_tensor)

        pred_np = pred.squeeze(0).cpu().numpy()  # Shape: [2, chunk_len]
        actual_len = end - start
        output[:, start:end] = pred_np[:, :actual_len]
    
    del inp_tensor, pred, pred_np
    if device == "cuda":
        torch.cuda.empty_cache()
    return output


# ============================================================
# MAIN ANALYSIS FUNCTION
# ============================================================

def run_residual_comparison():
    print("Starting Residual Comparison Analysis...")

    # 1. Validation & Paths
    sr = CONFIG["sample_rate"]
    train_subdir = CONFIG.get("train_subdir", "train")
    input_dir = PROJECT_ROOT / train_subdir / CONFIG.get("input_dir_name", "input")
    target_dir = PROJECT_ROOT / train_subdir / CONFIG.get("target_dir_name", "target")

    if not input_dir.exists() or not target_dir.exists():
        raise FileNotFoundError("Dataset folders missing. Ensure 'train/input' and 'train/target' exist.")

    input_files = set(f.stem for f in input_dir.glob("*.wav"))
    target_files = set(f.stem for f in target_dir.glob("*.wav"))
    paired = input_files & target_files
    if not paired:
        raise FileNotFoundError("No paired .wav files found in input/target folders.")

    song_name = TEST_FILENAME if TEST_FILENAME in paired else sorted(paired)[0]
    print(f"Selected file: {song_name}")

    # 2. Load Audio (Stereo)
    inp, _ = librosa.load(input_dir / f"{song_name}.wav", sr=sr, mono=False)
    tgt, _ = librosa.load(target_dir / f"{song_name}.wav", sr=sr, mono=False)
    if inp.ndim == 1: inp = np.vstack([inp, inp])
    if tgt.ndim == 1: tgt = np.vstack([tgt, tgt])

    # 3. Align & Compute Ground Truth Residual
    print("\nAligning & Computing Ground Truth...")
    inp_rms, tgt_rms = match_rms(inp, tgt)
    a_inp, a_tgt, _ = align_audio_by_cross_correlation(inp_rms, tgt_rms, sr)
    gt_residual = a_tgt - a_inp
    print(f"   GT Residual Shape: {gt_residual.shape}")

    # 4. Load Model & Predict
    print("\nLoading Model & Running Inference...")
    chk_cfg = CONFIG.get("checkpointing", {"downsample":[True]*4, "bottleneck":True, "upsample":[True]*4})
    model = IndahModel(base_channels=CONFIG.get("base_channels", 48), checkpointing_config=chk_cfg).to(DEVICE)

    if CHECKPOINT_PATH.exists():
        ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Weights loaded from {CHECKPOINT_NAME}")
    else:
        print("No checkpoint found. Using random init (metrics will be low).")

    pred_residual = predict_residual_stereo(model, a_inp, DEVICE, chunk_sec=CONFIG.get("chunk_duration_sec", 8))
    print(f"   Predicted Residual Shape: {pred_residual.shape}")

    # 5. Match Lengths & Compute Metrics
    min_len = min(gt_residual.shape[1], pred_residual.shape[1])
    gt_res = gt_residual[:, :min_len]
    pred_res = pred_residual[:, :min_len]

    gt_flat = gt_res.flatten()
    pred_flat = pred_res.flatten()

    corr = np.corrcoef(gt_flat, pred_flat)[0,1]
    mse = np.mean((gt_flat - pred_flat)**2)
    energy_ratio = np.sum(pred_flat**2) / (np.sum(gt_flat**2) + 1e-9)

    print("\nEvaluation Metrics:")
    print(f"   Correlation:      {corr:.4f}")
    print(f"   MSE:              {mse:.6f}")
    print(f"   Energy Ratio:     {energy_ratio:.2f}x")
    
    if corr > 0.6: print("Excellent: Model captures residual dynamics accurately.")
    elif corr > 0.3: print("Moderate: Model learns general patterns but lacks detail.")
    else: print("Poor: Model prediction does not correlate with ground truth.")

    # 6. Save Results
    sf.write(OUTPUT_DIR / f"{song_name}_gt_residual.wav", gt_res.T, sr)
    sf.write(OUTPUT_DIR / f"{song_name}_pred_residual.wav", pred_res.T, sr)
    reconstructed = a_inp[:, :min_len] + pred_residual
    sf.write(OUTPUT_DIR / f"{song_name}_reconstructed.wav", reconstructed.T, sr)

    # 7. Plotting
    print("\nGenerating Comparison Plots...")
    plot_residual_comparison(gt_res, pred_res, song_name, sr, corr, mse)
    print("Residual Analysis Complete! Check 'analysis_output/' folder.")

def plot_residual_comparison(gt, pred, name, sr, corr, mse):
    limit = min(5 * sr, gt.shape[1])
    t = np.arange(limit) / sr
    step = 50

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # 1. Waveform Comparison
    axes[0,0].plot(t, gt[0, :limit], label='Ground Truth', alpha=0.8)
    axes[0,0].plot(t, pred[0, :limit], label='Prediction', alpha=0.6, linestyle='--')
    axes[0,0].set_title(f'Residual Waveform (Left Ch) - {name}')
    axes[0,0].legend()

    # 2. Scatter Plot
    axes[0,1].scatter(gt[0, :limit:step], pred[0, :limit:step], alpha=0.1, s=2)
    axes[0,1].plot(axes[0,1].get_xlim(), axes[0,1].get_xlim(), 'r--')
    axes[0,1].set_title(f'Scatter: GT vs Pred (Corr: {corr:.2f})')
    axes[0,1].set_xlabel('Ground Truth')
    axes[0,1].set_ylabel('Prediction')

    # 3. Error Distribution
    error = (gt - pred).flatten()
    axes[1,0].hist(error, bins=100, color='teal', alpha=0.7)
    axes[1,0].set_title(f'Prediction Error Distribution (MSE: {mse:.4f})')
    axes[1,0].set_xlabel('Error Value')

    # 4. Spectrogram (Ground Truth)
    D_gt = librosa.amplitude_to_db(np.abs(librosa.stft(gt[0, :limit])), ref=np.max)
    im = librosa.display.specshow(D_gt, sr=sr, x_axis='time', ax=axes[1,1])
    axes[1,1].set_title('Ground Truth Residual Spectrogram')
    plt.colorbar(im, ax=axes[1,1])

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}_residual_comparison.png", dpi=150)
    plt.show()

# Execute
if __name__ == "__main__":
    run_residual_comparison()